In [13]:
import os
import sqlite3
import json
import pandas as pd
from bs4 import BeautifulSoup

# Use current working directory in Jupyter Notebooks
BASE_DIR = os.getcwd()

STUDENT_ID = "EYOUTH-30903300103337"

# Connect to the database using the absolute file path
db_path = os.path.join(BASE_DIR, "level_3_final_project_library.db")  # Corrected database file name

conn = sqlite3.connect(db_path)

# --- PART 2: Data Combination (Stages 1-3) ---
members_df = pd.read_sql_query("SELECT * FROM members", conn)
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts", conn)

db_counts = checkouts_df.groupby('member_id').size().reset_index(name='total_db_checkouts_per_member')
members_df = members_df.merge(db_counts, on='member_id', how='left')
members_df['total_db_checkouts_per_member'] = members_df['total_db_checkouts_per_member'].fillna(0).astype(int)

stage1_df = checkouts_df.merge(members_df, on='member_id', how='left')

# Stage 2: Fold in JSON Book Catalog
json_path = os.path.join(BASE_DIR, "level_3_final_project_book_catalog.json") # Corrected JSON catalog file name

with open(json_path, 'r', encoding='utf-8') as f:
    catalog_data = json.load(f)
catalog_df = pd.DataFrame(catalog_data)

db_books_df = pd.read_sql_query("SELECT * FROM books", conn)
full_catalog_df = db_books_df.merge(catalog_df, on='book_id', how='left')

stage2_df = stage1_df.merge(full_catalog_df, on='book_id', how='left')
stage2_df['checkout_source'] = 'Database'

# Stage 3: Parse & Add HTML Reading Kickoff Data
html_path = os.path.join(BASE_DIR, "level_3_final_project_event_signup.html") # Corrected HTML file name

with open(html_path, 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

table = soup.find('table')
rows = []
for tr in table.find_all('tr')[1:]:
    cols = [td.text.strip() for td in tr.find_all('td')]
    if cols:
        rows.append({
            'member_id': int(cols[0]),
            'book_id': int(cols[1]),
            'checkout_date': cols[2]
        })

kickoff_df = pd.DataFrame(rows)
kickoff_df['checkout_id'] = None
kickoff_df['return_date'] = None
kickoff_df['checkout_source'] = 'Reading Kickoff Web Page'

kickoff_merged = kickoff_df.merge(members_df, on='member_id', how='left')
kickoff_merged = kickoff_merged.merge(full_catalog_df, on='book_id', how='left')

final_combined_df = pd.concat([stage2_df, kickoff_merged], ignore_index=True)

# Save output CSV file in the project folder
output_csv = os.path.join(BASE_DIR, f"{STUDENT_ID}-Library.csv")
final_combined_df.to_csv(output_csv, index=False)

print(f"Success! Combined dataset saved to {output_csv} with {len(final_combined_df)} total records.")
conn.close()

import os
import pandas as pd

STUDENT_ID = "EYOUTH-30903300103337"

# Use current working directory in Jupyter Notebooks
BASE_DIR = os.getcwd()
input_csv = os.path.join(BASE_DIR, f"{STUDENT_ID}-Library.csv")

if not os.path.exists(input_csv):
    # Fallback search for any generated CSV if ID isn't replaced yet
    csv_files = [f for f in os.listdir(BASE_DIR) if f.endswith('.csv') and not f.endswith('-Cleaned.csv')]
    if csv_files:
        input_csv = os.path.join(BASE_DIR, csv_files[0])

df = pd.read_csv(input_csv)
print(f"Loaded initial dataset: {len(df)} rows.")

# --- PROBLEM 1: Missing Values (Imputation & Handling) ---
# Check missing counts
print("\n--- Problem 1: Missing Values ---")
print(df.isnull().sum())

# Decision Logic:
# 1. checkout_id: Missing in Reading Kickoff rows (26 rows). Impute negative IDs (-1, -2...) to distinguish from DB checkouts.
kickoff_mask = df['checkout_source'] == 'Reading Kickoff Web Page'
df.loc[kickoff_mask, 'checkout_id'] = [-i for i in range(1, kickoff_mask.sum() + 1)]

# 2. return_date: Missing in Kickoff rows (26 rows) and unreturned DB checkouts. Impute 'Not Returned / Summer Event'.
df['return_date'] = df['return_date'].fillna('Not Returned / Summer Event')

# 3. publication_year: Missing in 3 catalog books (book_ids with null years). Impute 'Unknown'.
df['publication_year'] = df['publication_year'].fillna('Unknown')

# --- PROBLEM 2: Duplicate Records ---
print("\n--- Problem 2: Duplicate Records ---")
# Identify true duplicates (ignoring index/sources if exact same event)
dup_cols = ['member_id', 'book_id', 'checkout_date', 'checkout_source']
true_dups = df[df.duplicated(subset=dup_cols, keep='first')]
print(f"True duplicate checkouts found: {len(true_dups)}")

df = df.drop_duplicates(subset=dup_cols, keep='first').copy()

# --- PROBLEM 3: Text Standardization ---
print("\n--- Problem 3: Text Inconsistencies ---")
# Neighborhood standardization
if 'neighborhood' in df.columns:
    print("Neighborhoods before:", df['neighborhood'].unique())
    df['neighborhood'] = df['neighborhood'].astype(str).str.strip().str.title()
    # Fix specific typos/casing
    df['neighborhood'] = df['neighborhood'].replace({
        'Heliopolis': 'Heliopolis',
        'Heliopolis': 'Heliopolis',
        'Nasr  City': 'Nasr City',
        'Nasr City': 'Nasr City',
        'Zamalek': 'Zamalek',
        'Maadi ': 'Maadi',
        'Maadi': 'Maadi'
    })
    print("Neighborhoods after:", df['neighborhood'].unique())

# Membership status standardization
if 'membership_status' in df.columns:
    print("Statuses before:", df['membership_status'].unique())
    df['membership_status'] = df['membership_status'].astype(str).str.strip().str.capitalize()
    print("Statuses after:", df['membership_status'].unique())

# --- PROBLEM 4: Orphan / Unregistered Member IDs ---
print("\n--- Problem 4: Orphan Member IDs ---")
# Identify member_ids that do not exist in the registered members list (first_name is null/missing)
orphan_rows = df[df['first_name'].isnull()]
print(f"Orphan checkout records found: {len(orphan_rows)}")

# Decision: Drop orphan checkout records to maintain database integrity (or flag if required)
df = df[df['first_name'].notnull()].copy()

# Save cleaned output
output_cleaned_csv = os.path.join(BASE_DIR, f"{STUDENT_ID}-Library-Cleaned.csv")
df.to_csv(output_cleaned_csv, index=False)
print(f"\nCleaned dataset saved successfully to {output_cleaned_csv} with {len(df)} total valid records!")

import os
import pandas as pd

STUDENT_ID = "EYOUTH-30903300103337"

# Use current working directory for Jupyter Notebooks
BASE_DIR = os.getcwd()
cleaned_csv = os.path.join(BASE_DIR, f"{STUDENT_ID}-Library-Cleaned.csv")

if not os.path.exists(cleaned_csv):
    # Fallback search for cleaned dataset
    csv_files = [f for f in os.listdir(BASE_DIR) if f.endswith('-Cleaned.csv')]
    if csv_files:
        cleaned_csv = os.path.join(BASE_DIR, csv_files[0])

df = pd.read_csv(cleaned_csv)

# Aggregate Members and Checkouts per Neighborhood
member_counts = df.groupby('neighborhood')['member_id'].nunique().reset_index(name='member_count')
checkout_counts = df.groupby('neighborhood').size().reset_index(name='checkout_count')

fairness_df = member_counts.merge(checkout_counts, on='neighborhood')
fairness_df['checkouts_per_member'] = (fairness_df['checkout_count'] / fairness_df['member_count']).round(2)
fairness_df['member_share_%'] = ((fairness_df['member_count'] / fairness_df['member_count'].sum()) * 100).round(2)
fairness_df['checkout_share_%'] = ((fairness_df['checkout_count'] / fairness_df['checkout_count'].sum()) * 100).round(2)

print("--- NEIGHBORHOOD FAIRNESS SUMMARY ---")
print(fairness_df.to_string(index=False))

Success! Combined dataset saved to /content/EYOUTH-30903300103337-Library.csv with 417 total records.
Loaded initial dataset: 417 rows.

--- Problem 1: Missing Values ---
checkout_id                      26
member_id                         0
book_id                           0
checkout_date                     0
return_date                      91
first_name                        5
last_name                         5
grade                            41
neighborhood                      5
membership_status                 5
join_date                        11
total_db_checkouts_per_member     5
title                             0
author                            0
genre                             0
pages                             0
publication_year                 35
publisher                         0
checkout_source                   0
dtype: int64

--- Problem 2: Duplicate Records ---
True duplicate checkouts found: 8

--- Problem 3: Text Inconsistencies ---
Neighborhoods befor